# Genomics: gene-panel expression explorer (GTEx v10)

**For:** genetics and rare disease, molecular genetics, neuro-oncology and neurology, bioinformatics.

**Questions:**
1. For a candidate gene panel, in which tissues is each gene expressed, and how tissue-specific is it?
2. In a tissue of interest, does expression vary with donor age or sex?

**Data:** open-access GTEx v10 (bulk RNA-seq, 50+ tissues, postmortem adult donors aged 20–79).
Files are read in place from the `adult-gtex` bucket. Nothing is copied into the workspace.

In [ ]:
import gzip, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import storage

gcs = storage.Client().bucket("adult-gtex")

def read_gct(path, genes=None):
    """Read a GCT file from the GTEx bucket, optionally keeping only some gene symbols."""
    raw = gzip.decompress(gcs.blob(path).download_as_bytes())
    df = pd.read_csv(io.BytesIO(raw), sep="\t", skiprows=2)
    return df[df.Description.isin(genes)] if genes is not None else df

## Pick a panel (edit freely)

In [ ]:
PANELS = {
    "pediatric_glioma":        ["H3-3A", "H3C2", "ACVR1", "PDGFRA", "BRAF", "TP53", "ATRX", "PPM1D", "NF1", "FGFR1", "IDH1", "MYCN"],
    "epileptic_encephalopathy": ["SCN1A", "SCN2A", "KCNQ2", "STXBP1", "CDKL5", "SLC2A1", "MECP2", "SYNGAP1", "GRIN2B", "PCDH19"],
    "ciliopathy":              ["BBS1", "BBS10", "CEP290", "ARL13B", "IFT88", "MKS1", "NPHP1", "TMEM67", "AHI1", "PKD1"],
    "rasopathy":               ["PTPN11", "SOS1", "RAF1", "KRAS", "NF1", "RIT1", "SHOC2", "BRAF", "MAP2K1", "LZTR1"],
}
PANEL = "pediatric_glioma"
TISSUE_FILE = "brain_cortex"      # any file in bulk-gex/v10/rna-seq/tpms-by-tissue/, e.g. heart_left_ventricle, liver
genes = PANELS[PANEL]

## 1. Median expression across all tissues + tissue specificity (τ)
τ = 0 for a gene expressed evenly everywhere and 1 for a gene expressed in a single tissue (Yanai et al. 2005).

In [ ]:
median = read_gct("bulk-gex/v10/rna-seq/GTEx_Analysis_v10_RNASeQCv2.4.2_gene_median_tpm.gct.gz", genes)
median = median.drop(columns="Name").groupby("Description").max().reindex(genes).dropna(how="all")
log = np.log2(median + 1)
tau = ((1 - log.div(log.max(axis=1), axis=0)).sum(axis=1) / (log.shape[1] - 1)).round(2)

order = log.columns[np.argsort([0 if c.startswith("Brain") else 1 for c in log.columns], kind="stable")]
fig, ax = plt.subplots(figsize=(15, 0.45 * len(log) + 2))
im = ax.imshow(log[order], aspect="auto", cmap="viridis")
ax.set_yticks(range(len(log)), [f"{g}  (τ={tau[g]})" for g in log.index])
ax.set_xticks(range(len(order)), [c.replace("_", " ") for c in order], rotation=80, ha="right", fontsize=7)
plt.colorbar(im, label="log2(median TPM + 1)", fraction=.02)
ax.set_title(f"{PANEL}: median expression across {len(order)} GTEx tissues (brain first)")
plt.tight_layout(); plt.show()

pd.DataFrame({"top tissue": median.idxmax(axis=1), "max median TPM": median.max(axis=1).round(1), "tau": tau})

## 2. Donor-level expression in one tissue, by age bracket and sex
Join per-sample TPMs to the open subject phenotypes (sex, 10-year age bracket).

In [ ]:
tpm = read_gct(f"bulk-gex/v10/rna-seq/tpms-by-tissue/gene_tpm_v10_{TISSUE_FILE}.gct.gz", genes)
long = (tpm.drop(columns="Name").melt(id_vars="Description", var_name="sample", value_name="tpm")
           .rename(columns={"Description": "gene"}))
long["SUBJID"] = long["sample"].str.split("-").str[:2].str.join("-")

pheno = pd.read_csv(io.BytesIO(gcs.blob(
    "annotations/v10/metadata-files/GTEx_Analysis_v10_Annotations_SubjectPhenotypesDS.txt").download_as_bytes()), sep="\t")
pheno["sex"] = pheno.SEX.map({1: "Male", 2: "Female"})
long = long.merge(pheno[["SUBJID", "AGE", "sex"]], on="SUBJID")
print(f"{TISSUE_FILE}: {long['sample'].nunique()} samples from {long.SUBJID.nunique()} donors")

show = [g for g in genes if g in long.gene.unique()][:6]
ages = sorted(long.AGE.unique())
fig, axes = plt.subplots(1, len(show), figsize=(3.2 * len(show), 3.5), sharey=False)
for ax, g in zip(np.atleast_1d(axes), show):
    d = long[long.gene == g]
    for i, (sex, color) in enumerate([("Female", "#c0504d"), ("Male", "#3b6ea5")]):
        data = [np.log2(d[(d.AGE == a) & (d.sex == sex)].tpm + 1) for a in ages]
        bp = ax.boxplot(data, positions=np.arange(len(ages)) * 2.5 + i, widths=.8, patch_artist=True, showfliers=False)
        for b in bp["boxes"]:
            b.set_facecolor(color); b.set_alpha(.5)
    ax.set_xticks(np.arange(len(ages)) * 2.5 + .5, ages, rotation=45, fontsize=7)
    ax.set_title(g); ax.spines[["top", "right"]].set_visible(False)
np.atleast_1d(axes)[0].set_ylabel("log2(TPM + 1)")
fig.suptitle(f"{TISSUE_FILE.replace('_', ' ')}: expression by donor age (red = female, blue = male)")
plt.tight_layout(); plt.show()

In [ ]:
from scipy.stats import spearmanr
age_mid = long.AGE.str[:2].astype(int) + 5
rows = []
for g, d in long.assign(age_mid=age_mid).groupby("gene"):
    rho, p = spearmanr(d.age_mid, d.tpm)
    rows.append({"gene": g, "spearman_rho_vs_age": round(rho, 2), "p": f"{p:.1e}",
                 "median_TPM_female": round(d[d.sex == "Female"].tpm.median(), 1),
                 "median_TPM_male": round(d[d.sex == "Male"].tpm.median(), 1)})
pd.DataFrame(rows).set_index("gene").reindex(show)

**Go further (bioinformatics):**
- eQTL / sQTL summary statistics: `bulk-qtl/v10/single-tissue-cis-qtl/` (tar archives, 2–20 GB, so
  extract on a larger app or in a workflow). Fine-mapped SuSiE credible sets: `bulk-qtl/v10/susie-qtl/`.
- Single-cell and long-read RNA: `single-cell/`, `long-read-data/v9/`.
- Use `adult-gtex-manifest.files` in BigQuery to search every file by name or path without listing the bucket.
- Heavy pipelines (nf-core/rnaseq, sarek, etc.) run on the Workbench Nextflow / dsub engines. See the
  July demo for a variant-calling example.

## Provenance

Generate information about this notebook environment and the packages installed.

In [ ]:
!date

Conda and pip installed packages:

In [ ]:
!conda env export

JupyterLab extensions:

In [ ]:
!jupyter labextension list

Number of cores:

In [ ]:
!grep ^processor /proc/cpuinfo | wc -l

Memory:

In [ ]:
!grep "^MemTotal:" /proc/meminfo

---
Copyright 2026 Verily Life Sciences LLC

Use of this source code is governed by a BSD-style
license that can be found in the LICENSE file or at
https://developers.google.com/open-source/licenses/bsd